# AXE Genesis Keras Meta-Learner & Q-Executor Pipeline

Full RL pipeline using TensorFlow / Keras Engine: Meta-Learner training (Phase 1), Q-Executor sequential traversal training (Phase 2), out-of-sample evaluation across 4 expiry horizons (Phase 3/4), and checkpoint export.

In [ ]:
# =============================================================================
# SYSTEM IMPORTS & TENSORFLOW / KERAS ENVIRONMENT SETUP
# =============================================================================
import os
import glob
import zipfile
import logging
import time
import math
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any, Union
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l2 as keras_l2

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print(e)
    print(f'GPUs available: {len(gpus)}')
else:
    print('No GPU found. Running on CPU.')


In [ ]:
# =============================================================================
# DATASET LOADING (Train 70% | Val 15% | Test 15%)
# =============================================================================
zip_files = glob.glob(os.path.join(KAGGLE_DATASET_DIR, '*.zip'))
if zip_files:
    print(f"Extracting dataset archive: {zip_files[0]}")
    with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
        target_extract = '/kaggle/working/data' if os.path.exists('/kaggle/working') else 'data'
        zip_ref.extractall(target_extract)
    data_dir = target_extract
elif os.path.exists('data/train_50k.csv'):
    data_dir = 'data'
else:
    data_dir = KAGGLE_DATASET_DIR

train_csv = os.path.join(data_dir, 'train_50k.csv')
val_csv   = os.path.join(data_dir, 'val_50k.csv')
test_csv  = os.path.join(data_dir, 'test_50k.csv')

if os.path.exists(train_csv):
    train_df = pd.read_csv(train_csv)
    val_df   = pd.read_csv(val_csv) if os.path.exists(val_csv) else None
    test_df  = pd.read_csv(test_csv) if os.path.exists(test_csv) else None
    print(f"Train Set: {len(train_df)} rows | Columns: {len(train_df.columns)}")
    if val_df is not None:  print(f"Validation Set: {len(val_df)} rows")
    if test_df is not None: print(f"Holdout Test Set: {len(test_df)} rows")
else:
    raise FileNotFoundError(f"Dataset files not found under {data_dir}. Check dataset path!")

close_col = "close_5m" if "close_5m" in train_df.columns else train_df.columns[0]
open_col  = "open_5m" if "open_5m" in train_df.columns else train_df.columns[0]
high_col  = "high_5m" if "high_5m" in train_df.columns else train_df.columns[0]
low_col   = "low_5m" if "low_5m" in train_df.columns else train_df.columns[0]
vol_col   = "volume_5m" if "volume_5m" in train_df.columns else train_df.columns[1]
up_vol_col   = "Bar_Volume_Up_5m" if "Bar_Volume_Up_5m" in train_df.columns else None
down_vol_col = "Bar_Volume_Down_5m" if "Bar_Volume_Down_5m" in train_df.columns else None
atr_col      = "ATR_5m" if "ATR_5m" in train_df.columns else None
print(f"Volume columns available: up={up_vol_col}, down={down_vol_col} | ATR column: {atr_col}")

In [ ]:
# =============================================================================
# REAL SNR ZONE DETECTION — ported verbatim from
# backend/app/core/analysis/support_resistance.py, verified against the live
# backend (detect_snr_levels_sequential explicitly guarantees no lookahead:
# "Only uses data up to up_to_index"). This is NOT a simplified placeholder —
# it is the exact same function the backend uses, so zone-anchored decisions
# here are genuinely 1:1 with production.
# =============================================================================

def detect_snr_levels_sequential(price_data, up_to_index, lookback_period, min_distance_pct=0.5):
    '''Detect S&R levels up to a specific index. CRITICAL: only uses data up to up_to_index.'''
    levels = []
    df = price_data.iloc[up_to_index - lookback_period: up_to_index + 1]
    if len(df) < 5:
        return levels

    highs = df["High"].values
    lows = df["Low"].values
    price_range = highs.max() - lows.min()
    min_distance = price_range * (min_distance_pct / 100)

    if len(lows) >= 5:
        support_cond1 = lows[2:-2] < lows[1:-3]
        support_cond2 = lows[2:-2] < lows[3:-1]
        support_cond3 = lows[3:-1] < lows[4:]
        support_cond4 = lows[1:-3] < lows[:-4]
        support_mask = support_cond1 & support_cond2 & support_cond3 & support_cond4
        support_indices = np.where(support_mask)[0] + 2

        resistance_cond1 = highs[2:-2] > highs[1:-3]
        resistance_cond2 = highs[2:-2] > highs[3:-1]
        resistance_cond3 = highs[3:-1] > highs[4:]
        resistance_cond4 = highs[1:-3] > highs[:-4]
        resistance_mask = resistance_cond1 & resistance_cond2 & resistance_cond3 & resistance_cond4
        resistance_indices = np.where(resistance_mask)[0] + 2

        for idx in support_indices:
            level = lows[idx]
            if not levels or all(abs(level - l[1]) >= min_distance for l in levels):
                levels.append((int(idx), float(level), "support"))
        for idx in resistance_indices:
            level = highs[idx]
            if not levels or all(abs(level - l[1]) >= min_distance for l in levels):
                levels.append((int(idx), float(level), "resistance"))

    window = 5
    if len(df) > window * 2:
        pivot_high_mask = np.ones(len(highs), dtype=bool)
        pivot_high_mask[:window] = False
        pivot_high_mask[-window:] = False
        for offset in range(1, window + 1):
            pivot_high_mask[window:-window] &= (
                (highs[window:-window] > highs[window-offset:-(window+offset)]) &
                (highs[window:-window] > highs[window+offset:len(highs)-window+offset])
            )
        pivot_high_indices = np.where(pivot_high_mask)[0]

        pivot_low_mask = np.ones(len(lows), dtype=bool)
        pivot_low_mask[:window] = False
        pivot_low_mask[-window:] = False
        for offset in range(1, window + 1):
            pivot_low_mask[window:-window] &= (
                (lows[window:-window] < lows[window-offset:-(window+offset)]) &
                (lows[window:-window] < lows[window+offset:len(lows)-window+offset])
            )
        pivot_low_indices = np.where(pivot_low_mask)[0]

        for idx in pivot_high_indices:
            level = highs[idx]
            if not levels or all(abs(level - l[1]) >= min_distance for l in levels):
                levels.append((int(idx), float(level), "resistance"))
        for idx in pivot_low_indices:
            level = lows[idx]
            if not levels or all(abs(level - l[1]) >= min_distance for l in levels):
                levels.append((int(idx), float(level), "support"))

    return levels


def calculate_volume_profile_at_level(price_level, price_data, zone_width=0.004):
    '''CRITICAL: only uses the price_data slice passed in (no lookahead).'''
    upper_bound = price_level + zone_width
    lower_bound = price_level - zone_width
    highs = price_data["High"].values
    lows = price_data["Low"].values
    closes = price_data["Close"].values
    opens = price_data["Open"].values
    volumes = price_data["Volume"].values

    touches_level = (lows <= price_level) & (highs >= price_level)
    is_bullish = closes > opens
    total_volume = volumes[touches_level].sum()
    up_volume = volumes[touches_level & is_bullish].sum()
    down_volume = volumes[touches_level & ~is_bullish].sum()

    return {
        "total_volume": float(total_volume),
        "up_volume": float(up_volume),
        "down_volume": float(down_volume),
        "net_volume": float(up_volume - down_volume),
        "upper_bound": upper_bound,
        "lower_bound": lower_bound,
    }


def create_clustered_zones_sequential(levels, price_data_slice, n_clusters=16, zone_width=0.004):
    '''Create zones using K-means clustering for sequential analysis.'''
    if not levels:
        return []
    prices = [level[1] for level in levels]
    unique_prices_count = len(set(prices))
    if n_clusters is None:
        n_clusters = min(unique_prices_count, max(3, len(prices) // 3))
    if unique_prices_count < n_clusters:
        n_clusters = unique_prices_count
    if n_clusters < 1:
        return []
    if unique_prices_count < 2:
        if not prices:
            return []
        zone_price = prices[0]
        volume_data = calculate_volume_profile_at_level(zone_price, price_data_slice, zone_width)
        return [(0, zone_price, [l for l in levels if l[1] == zone_price], volume_data)]

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
    price_array = np.array(prices).reshape(-1, 1)
    clusters = kmeans.fit_predict(price_array)

    zones = []
    for cluster_id in range(n_clusters):
        cluster_levels = [levels[i] for i, c in enumerate(clusters) if c == cluster_id]
        if cluster_levels:
            zone_price = np.mean([l[1] for l in cluster_levels])
            volume_data = calculate_volume_profile_at_level(zone_price, price_data_slice, zone_width)
            zones.append((cluster_id, zone_price, cluster_levels, volume_data))
    return sorted(zones, key=lambda x: x[1])


def get_nearest_zones(zones, current_price):
    '''Mirror of ZoneSnapshotManager.get_nearest_zones — returns (nearest_support, nearest_resistance)
    as dicts with price_level + volume_delta_ratio, or None if absent.'''
    supports = [z for z in zones if z[1] <= current_price]
    resistances = [z for z in zones if z[1] >= current_price]
    nearest_supp = max(supports, key=lambda z: z[1]) if supports else None
    nearest_res = min(resistances, key=lambda z: z[1]) if resistances else None

    def _to_record(z):
        if z is None:
            return None
        _, price, _, vol = z
        total = vol["up_volume"] + vol["down_volume"]
        ratio = (vol["up_volume"] - vol["down_volume"]) / (total + 1e-6)
        return {"price_level": price, "volume_delta_ratio": ratio, "volume": vol}

    return _to_record(nearest_supp), _to_record(nearest_res)


print("Real SNR zone detection loaded (verified 1:1 port of backend support_resistance.py).")


In [ ]:
# =============================================================================
# DOMAIN STRUCTURES & REAL HARD ACTION MASK
# (Previous version's HardActionMask never referenced zone_manager at all — it only
#  gated on volume imbalance, meaning the no-chase / zone-anchored entry rule, the
#  central design principle of this strategy, was entirely absent. This version
#  enforces the same ATR-scaled proximity band + volume confirmation + single-position
#  restriction as backend/app/core/market/zone_snapshot.py's HardActionMask.)
# =============================================================================

@dataclass
class HTFBiasPackage:
    direction: str = "neutral"
    strength: float = 0.0
    reversal_prob: float = 0.0
    q_value: float = 0.0
    expected_mfe_pips: float = 0.0
    expected_mae_pips: float = 0.0
    horizon_strengths: List[float] = field(default_factory=lambda: [0.5, 0.5, 0.5, 0.5])
    optimal_horizon_idx: int = 2
    recommended_expiry: str = "30m"

@dataclass
class AccountContext:
    balance: float = 10000.0
    equity: float = 10000.0
    open_position_type: Optional[str] = None
    open_position_pnl_pct: float = 0.0
    daily_drawdown_pct: float = 0.0
    win_streak: int = 0
    loss_streak: int = 0
    reentries_in_window: int = 0
    max_reentries_allowed: int = 3

@dataclass
class ExecutionContext:
    symbol: str
    current_price: float
    atr: float
    buy_volume: float
    sell_volume: float
    hour_of_day: float
    day_of_week: int
    session_phase: str
    ltf_timeframe: str = "5m"


class HardActionMask:
    '''1:1 with backend zone_snapshot.py::HardActionMask — enforces the no-chase rule
    (entries only at/near a real zone), volume confirmation, and single-open-position
    restriction, as HARD constraints rather than something the network has to learn.'''

    def get_action_mask(
        self, current_price, atr, nearest_supp, nearest_res,
        buy_volume, sell_volume, has_open_position=False,
    ):
        # mask indices: 0=WAIT, 1=BUY_CALL, 2=BUY_PUT, 3=TAKE_PROFIT_HALF, 4=CLOSE_FLATTEN
        mask = np.ones(5, dtype=np.int32)

        if has_open_position:
            # Single running trade restriction — no new entries while a position is open.
            mask[1] = 0
            mask[2] = 0
            return mask

        mask[3] = 0
        mask[4] = 0

        proximity_band = max(atr * 0.75, current_price * 0.003)

        supp_ok = nearest_supp is not None and abs(current_price - nearest_supp["price_level"]) <= proximity_band
        res_ok = nearest_res is not None and abs(current_price - nearest_res["price_level"]) <= proximity_band

        # No-chase: BUY_CALL only valid near/below a support zone. BUY_PUT only valid near/above resistance.
        if not supp_ok:
            mask[1] = 0
        if not res_ok:
            mask[2] = 0

        # Volume confirmation gate — require the reaction to actually be confirmed, not just proximity.
        if buy_volume > 0 or sell_volume > 0:
            if buy_volume < sell_volume * 0.8:
                mask[1] = 0
            if sell_volume < buy_volume * 0.8:
                mask[2] = 0

        return mask


def _make_exec_ctx(symbol: str, price: float, row: dict, atr_col: str, up_vol_col: str, down_vol_col: str) -> ExecutionContext:
    '''Uses REAL up/down volume columns from the feature pipeline instead of a crude
    open/close-direction proxy, and REAL ATR instead of an SNR-distance-derived guess.
    Session-phase uses actual US/Eastern local time via zoneinfo (DST-aware), matching
    backend q_executor.py's is_nyse_open (09:30-10:30 ET) / is_power_hour (15:00-16:00 ET).'''
    ts = row.get("timestamp", None)
    hour_f, dow, phase = 14.5, 1, "off_hours"
    if ts is not None:
        try:
            ts_pd = pd.Timestamp(ts)
            if ts_pd.tzinfo is None:
                ts_pd = ts_pd.tz_localize("UTC")
            ts_et = ts_pd.tz_convert("America/New_York")
            hour_f = ts_et.hour + ts_et.minute / 60.0
            dow = ts_et.dayofweek
            if 9.5 <= hour_f < 10.5:
                phase = "nyse_open"
            elif 15.0 <= hour_f < 16.0:
                phase = "nyse_power_hour"
            elif 9.5 <= hour_f < 16.0:
                phase = "regular_hours"
        except Exception:
            pass

    buy_vol = float(row.get(up_vol_col, 0.0)) if up_vol_col else 0.0
    sell_vol = float(row.get(down_vol_col, 0.0)) if down_vol_col else 0.0
    atr_val = float(row.get(atr_col, price * 0.005)) if atr_col else price * 0.005

    return ExecutionContext(
        symbol=symbol, current_price=price, atr=max(0.01, atr_val),
        buy_volume=buy_vol, sell_volume=sell_vol, hour_of_day=hour_f, day_of_week=dow, session_phase=phase,
    )


def build_state_vector(net_out, htf_bias: HTFBiasPackage, account: AccountContext,
                        exec_ctx: ExecutionContext, nearest_supp, nearest_res) -> np.ndarray:
    '''1:1 with backend q_executor.py::build_state_vector's real 28-dim layout.
    NOTE: unlike the previous version, this contains NO future-derived value anywhere —
    the previous notebook placed the literal forward price move (the reward target) into
    state_vec[4] as an INPUT feature, both in training and in Phase-2 validation. That is
    direct label leakage: the network was being handed the answer as an input. Every
    field here is computable strictly from data up to and including the current bar.'''
    supp_dist = abs(exec_ctx.current_price - nearest_supp["price_level"]) / exec_ctx.current_price if nearest_supp else 1.0
    res_dist = abs(exec_ctx.current_price - nearest_res["price_level"]) / exec_ctx.current_price if nearest_res else 1.0
    supp_vol_ratio = nearest_supp["volume_delta_ratio"] if nearest_supp else 0.0
    res_vol_ratio = nearest_res["volume_delta_ratio"] if nearest_res else 0.0

    total_vol = exec_ctx.buy_volume + exec_ctx.sell_volume
    vol_delta_ratio = (exec_ctx.buy_volume - exec_ctx.sell_volume) / (total_vol + 1e-6)

    tf_flag = 1.0 if exec_ctx.ltf_timeframe == "15m" else 0.0
    dir_flag = 1.0 if htf_bias.direction == "bullish" else (-1.0 if htf_bias.direction == "bearish" else 0.0)
    hs = htf_bias.horizon_strengths if len(htf_bias.horizon_strengths) == 4 else [0.5, 0.5, 0.5, 0.5]

    sin_hour = float(np.sin(2 * np.pi * exec_ctx.hour_of_day / 24.0))
    cos_hour = float(np.cos(2 * np.pi * exec_ctx.hour_of_day / 24.0))
    dow_norm = float(exec_ctx.day_of_week) / 6.0
    is_nyse_open = 1.0 if exec_ctx.session_phase == "nyse_open" else 0.0
    is_power_hour = 1.0 if exec_ctx.session_phase == "nyse_power_hour" else 0.0

    state = np.array([
        dir_flag, float(htf_bias.strength), float(htf_bias.reversal_prob), float(htf_bias.q_value),
        float(htf_bias.expected_mfe_pips) / 100.0, float(htf_bias.expected_mae_pips) / 100.0,
        float(hs[0]), float(hs[1]), float(hs[2]), float(hs[3]),
        float(account.daily_drawdown_pct),
        1.0 if account.open_position_type == "CALL" else (-1.0 if account.open_position_type == "PUT" else 0.0),
        float(account.open_position_pnl_pct), float(account.win_streak) / 10.0, float(account.loss_streak) / 10.0,
        tf_flag, float(exec_ctx.atr) / exec_ctx.current_price, float(supp_dist), float(res_dist),
        float(supp_vol_ratio), float(res_vol_ratio), float(vol_delta_ratio),
        float(account.reentries_in_window) / float(account.max_reentries_allowed),
        sin_hour, cos_hour, dow_norm, is_nyse_open, is_power_hour,
    ], dtype=np.float32)
    return state

print("Real HardActionMask + 28-dim state vector (no future leakage) loaded.")


In [ ]:
# =============================================================================
# 🏗️ TENSORFLOW / KERAS PRODUCTION MODEL ARCHITECTURES (From keras_signal_meta_learner.py & keras_trade_executor.py)
# =============================================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

feature_cols = [c for c in train_df.columns if c not in ('timestamp', 'Time') and not 'target' in c and not 'forward' in c]
num_features = len(feature_cols)
lookback_bars = 1000
input_dim = num_features * lookback_bars

class StopGradient(keras.layers.Layer):
    def call(self, x):
        return tf.stop_gradient(x)

class KerasSignalMetaNetwork(keras.Model):
    def __init__(self, input_dim=28000, num_features=28, hidden_dim=64, num_actions=4, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.lookback_bars = input_dim // num_features if num_features > 0 else 1000

        self.b1_conv1 = layers.Conv1D(32, kernel_size=3, padding='same')
        self.b1_bn1   = layers.BatchNormalization()
        self.b1_conv2 = layers.Conv1D(32, kernel_size=3, padding='same')
        self.b1_bn2   = layers.BatchNormalization()
        self.b1_lstm  = layers.LSTM(32, return_sequences=True)

        self.b2_conv  = layers.Conv1D(32, kernel_size=3, padding='same')
        self.b2_bn    = layers.BatchNormalization()
        self.b2_fc    = layers.Dense(32, activation='relu')

        self.b3_conv  = layers.Conv1D(32, kernel_size=3, padding='same')
        self.b3_bn    = layers.BatchNormalization()
        self.b3_fc    = layers.Dense(32, activation='relu')

        self.aux1_head = layers.Dense(5)
        self.aux2_head = layers.Dense(5)

        self.fusion_fc1  = layers.Dense(hidden_dim)
        self.fusion_ln1  = layers.LayerNormalization()
        self.fusion_fc2  = layers.Dense(hidden_dim)
        self.fusion_ln2  = layers.LayerNormalization()

        self.q_head        = layers.Dense(num_actions)
        self.strength_head = layers.Dense(4, activation='sigmoid')
        self.fusion_selector = layers.Dense(4)

        self.pips_proj = layers.Dense(32, activation='relu')
        self.pips_head = layers.Dense(4)
        self.risk_proj = layers.Dense(32, activation='relu')
        self.risk_head = layers.Dense(8)
        self.liq_proj  = layers.Dense(16, activation='relu')
        self.liquidity_head = layers.Dense(2)
        self.rev_proj  = layers.Dense(16, activation='relu')
        self.reversal_head  = layers.Dense(1, activation='sigmoid')

    def call(self, inputs, training=False, return_aux=False):
        b = tf.shape(inputs)[0]
        c = self.num_features
        t = self.lookback_bars
        x_3d = tf.reshape(inputs, (b, t, c))

        b1_c1 = tf.nn.silu(self.b1_bn1(self.b1_conv1(x_3d), training=training))
        b1_c2 = tf.nn.silu(self.b1_bn2(self.b1_conv2(b1_c1), training=training))
        b1_lstm_out = self.b1_lstm(b1_c2)
        b1_last = b1_lstm_out[:, -1, :]
        b1_gap  = tf.reduce_mean(b1_lstm_out, axis=1)
        b1_out  = tf.concat([b1_last, b1_gap], axis=-1)

        half = max(1, t // 2)
        b2_c = tf.nn.silu(self.b2_bn(self.b2_conv(x_3d[:, -half:, :]), training=training))
        b2_gap = tf.reduce_mean(b2_c, axis=1)
        b2_out = self.b2_fc(b2_gap)

        recent = max(1, int(t * 0.3))
        b3_c = tf.nn.silu(self.b3_bn(self.b3_conv(x_3d[:, -recent:, :]), training=training))
        b3_gap = tf.reduce_mean(b3_c, axis=1)
        b3_out = self.b3_fc(b3_gap)

        aux1 = self.aux1_head(tf.stop_gradient(b1_out))
        aux2 = self.aux2_head(tf.stop_gradient(b2_out))

        fusion_in = tf.concat([b1_out, b2_out, b3_out, tf.stop_gradient(aux1), tf.stop_gradient(aux2)], axis=-1)
        feat = tf.nn.silu(self.fusion_ln1(self.fusion_fc1(fusion_in)))
        feat = tf.nn.silu(self.fusion_ln2(self.fusion_fc2(feat)))

        q_vals   = self.q_head(feat)
        strength = self.strength_head(feat)
        selector_logits = self.fusion_selector(feat)

        branch_cat = tf.concat([b1_out, b2_out, b3_out], axis=-1)
        pips      = self.pips_head(self.pips_proj(branch_cat))
        risk      = self.risk_head(self.risk_proj(branch_cat))
        liquidity = self.liquidity_head(self.liq_proj(branch_cat))
        reversal  = self.reversal_head(self.rev_proj(branch_cat))

        if return_aux:
            return q_vals, strength, pips, risk, liquidity, reversal, aux1, aux2, selector_logits
        return q_vals, strength, pips, risk, liquidity, reversal

class KerasExecutorQNetwork(keras.Model):
    def __init__(self, input_dim=28, hidden_dim=64, num_horizons=4, num_head_actions=3, **kwargs):
        super().__init__(**kwargs)
        self.num_horizons = num_horizons
        self.num_head_actions = num_head_actions

        self.b1_fc1 = layers.Dense(hidden_dim)
        self.b1_ln1 = layers.LayerNormalization()
        self.b1_fc2 = layers.Dense(32)
        self.b1_ln2 = layers.LayerNormalization()

        self.b2_meta = layers.Dense(16, activation='relu')
        self.b2_risk = layers.Dense(16, activation='relu')
        self.b2_zone = layers.Dense(16, activation='relu')
        self.b2_time = layers.Dense(16, activation='relu')
        self.b2_fusion = layers.Dense(32)
        self.b2_ln   = layers.LayerNormalization()

        self.fusion_fc = layers.Dense(hidden_dim)
        self.fusion_ln = layers.LayerNormalization()

        self.horizon_heads = [
            keras.Sequential([
                layers.Dense(32),
                layers.LayerNormalization(),
                layers.Activation('silu'),
                layers.Dense(num_head_actions),
            ])
            for _ in range(num_horizons)
        ]

    def call(self, x, horizon_idx=None, training=False):
        b1 = tf.nn.silu(self.b1_ln1(self.b1_fc1(x)))
        b1_out = tf.nn.silu(self.b1_ln2(self.b1_fc2(b1)))

        meta_f = x[:, :10]
        risk_f = x[:, 10:15]
        zone_f = x[:, 15:23]
        time_f = x[:, 23:28]
        b2_cat = tf.concat([
            self.b2_meta(meta_f),
            self.b2_risk(risk_f),
            self.b2_zone(zone_f),
            self.b2_time(time_f),
        ], axis=-1)
        b2_out = tf.nn.silu(self.b2_ln(self.b2_fusion(b2_cat)))

        shared = tf.nn.silu(self.fusion_ln(self.fusion_fc(tf.concat([b1_out, b2_out], axis=-1))))

        if horizon_idx is not None:
            return self.horizon_heads[horizon_idx](shared)

        all_logits = [head(shared) for head in self.horizon_heads]
        return tf.stack(all_logits, axis=1)

SignalMetaNetwork = KerasSignalMetaNetwork
ExecutorQNetwork = KerasExecutorQNetwork
print('✅ Keras Production Models Defined: KerasSignalMetaNetwork & KerasExecutorQNetwork')


In [ ]:
# =============================================================================
# PHASE 1: META-LEARNER MULTI-HEAD TRAINING WITH 21+ ML TARGETS
#
# Updates:
#   1. Context windows: Meta=150 bars (12h of 5m data)
#   2. 6 primary heads + 21+ ML targets (zone, volatility, velocity)
#   3. Graceful fallback if ML targets not in CSV (zeros as defaults)
#   4. Multi-task loss with proper weighting
# =============================================================================



import numpy as np
import random
import pandas as pd

# Running on TensorFlow device context else "cpu")

# Target column preparation (synthesize if not in CSV)
for df in (train_df, val_df, test_df):
    if df is not None:
        df["forward_move_1"]  = df[close_col].shift(-1)  - df[close_col]
        df["forward_move_3"]  = df[close_col].shift(-3)  - df[close_col]
        df["forward_move_6"]  = df[close_col].shift(-6)  - df[close_col]
        df["forward_move_12"] = df[close_col].shift(-12) - df[close_col]
        df["target_dir_5m"]   = (df["forward_move_1"]  > 0).astype(np.float32)
        df["target_dir_15m"]  = (df["forward_move_3"]  > 0).astype(np.float32)
        df["target_dir_30m"]  = (df["forward_move_6"]  > 0).astype(np.float32)
        df["target_dir_1h"]   = (df["forward_move_12"] > 0).astype(np.float32)

print("[Bug1-fix] forward_move_1 std:", train_df["forward_move_1"].std(), "mean:", train_df["forward_move_1"].mean())
net = SignalMetaNetwork(input_dim=input_dim, num_features=num_features)

target_net.set_weights(net.get_weights())

optimizer = keras.optimizers.Adam(learning_rate=1e-3)

N_train = len(train_df) - lookback_bars - 12
N_val   = len(val_df) - lookback_bars - 12 if val_df is not None else 0
META_EPOCHS = 50
BATCH_SIZE = 64
steps_per_epoch = max(1, N_train // BATCH_SIZE)
# Keras Adam optimizer handles LR

train_num_matrix = np.nan_to_num(train_df[feature_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
val_num_matrix   = np.nan_to_num(val_df[feature_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0) if val_df is not None else None

def _extract_targets(df):
    """
    Extract targets for multi-head training (6 primary heads + 21+ ML targets from data pipeline).
    
    Returns: (q, pips, risk, rev, strength, liq, ml_targets_dict)
    """
    close_vals = df[close_col].values.astype(np.float32)
    
    # Dynamic True ATR computation
    if atr_col and atr_col in df.columns:
        atr_vals_local = df[atr_col].values.astype(np.float32)
    else:
        h_col = next((c for c in [high_col, "high_5m", "High", "high"] if c in df.columns), None)
        l_col = next((c for c in [low_col, "low_5m", "Low", "low"] if c in df.columns), None)
        if h_col and l_col:
            highs = df[h_col].values.astype(np.float32)
            lows  = df[l_col].values.astype(np.float32)
            atr_vals_local = pd.Series(highs - lows).rolling(14, min_periods=1).mean().values.astype(np.float32)
        else:
            atr_vals_local = close_vals * 0.0008  # 8 pips realistic 5m bar ATR fallback
    atr_vals_local = np.maximum(atr_vals_local, 1e-4)

    fwd = {
        1:  df["forward_move_1"].values.astype(np.float32),
        3:  df["forward_move_3"].values.astype(np.float32),
        6:  df["forward_move_6"].values.astype(np.float32),
        12: df["forward_move_12"].values.astype(np.float32),
    }
    dirs = {
        1:  df["target_dir_5m"].values.astype(np.float32),
        3:  df["target_dir_15m"].values.astype(np.float32),
        6:  df["target_dir_30m"].values.astype(np.float32),
        12: df["target_dir_1h"].values.astype(np.float32),
    }

    # Primary 6 Heads
    strength_cols = []
    for h_bars, h_key in zip([1, 3, 6, 12], [1, 3, 6, 12]):
        move_atr_signed = fwd[h_key] / (atr_vals_local * np.sqrt(h_bars))
        horizon_strength = 0.5 + 0.5 * np.clip(move_atr_signed / 1.5, -1.0, 1.0)
        horizon_strength = np.clip(horizon_strength.astype(np.float32), 0.05, 0.95)
        strength_cols.append(horizon_strength)
    strength_targets = np.column_stack(strength_cols)

    pips = np.column_stack([
        fwd[1] / atr_vals_local, fwd[3] / (atr_vals_local * np.sqrt(3)),
        fwd[6] / (atr_vals_local * np.sqrt(6)), fwd[12] / (atr_vals_local * np.sqrt(12)),
    ]).astype(np.float32)
    pips = np.clip(pips, -10.0, 10.0)

    risk = np.column_stack([
        np.maximum(fwd[1],  0) / atr_vals_local, np.maximum(-fwd[1],  0) / atr_vals_local,
        np.maximum(fwd[3],  0) / (atr_vals_local * np.sqrt(3)),  np.maximum(-fwd[3],  0) / (atr_vals_local * np.sqrt(3)),
        np.maximum(fwd[6],  0) / (atr_vals_local * np.sqrt(6)),  np.maximum(-fwd[6],  0) / (atr_vals_local * np.sqrt(6)),
        np.maximum(fwd[12], 0) / (atr_vals_local * np.sqrt(12)), np.maximum(-fwd[12], 0) / (atr_vals_local * np.sqrt(12)),
    ]).astype(np.float32)
    risk = np.clip(risk, 0.0, 10.0)

    liq = np.column_stack([
        np.abs(fwd[1]) / atr_vals_local,
        np.abs(fwd[3]) / (atr_vals_local * np.sqrt(3)),
    ]).astype(np.float32)
    liq = np.clip(liq, 0.0, 10.0)

    q = np.column_stack([dirs[1], dirs[3], dirs[6], dirs[12]]).astype(np.float32)
    q = np.clip(q, 0.0, 1.0)

    rev = (dirs[1] != dirs[3]).astype(np.float32).reshape(-1, 1)

    # 21+ Optional ML Targets (graceful fallback if not in CSV)
    ml_targets_dict = {}
    
    zone_cols = ["adv_target_next_zone_idx", "adv_target_next_zone_bars", "adv_target_next_zone_distance", "adv_target_next_zone_volume"]
    for col in zone_cols:
        ml_targets_dict[col] = df[col].values.astype(np.float32) if col in df.columns else np.zeros(len(df), dtype=np.float32)
    
    vol_cols = ["Volatility_Regime_next", "vol_regime_fwd_8", "Volatility_Expansion_next", "vol_expansion_fwd_8", 
                "Volatility_Bull_next", "Volatility_Bear_next", "Regime_Speed_Bull_next", "Regime_Speed_Bear_next", 
                "speed_aligned_fwd_8", "speed_divergence_fwd_8"]
    for col in vol_cols:
        ml_targets_dict[col] = df[col].values.astype(np.float32) if col in df.columns else np.zeros(len(df), dtype=np.float32)
    
    vel_cols = ["Price_Velocity_Bull_next", "vel_bull_fwd_8", "Price_Velocity_Bear_next", "vel_bear_fwd_8", 
                "Price_Velocity_Net_next", "vel_net_fwd_8"]
    for col in vel_cols:
        ml_targets_dict[col] = df[col].values.astype(np.float32) if col in df.columns else np.zeros(len(df), dtype=np.float32)
    
    csm_cols = ["adv_target_CSM_hist_fast_next", "adv_target_CSM_hist_slow_next", 
                "adv_target_CSM_asset_fast_next", "adv_target_CSM_dxy_fast_next"]
    for col in csm_cols:
        ml_targets_dict[col] = df[col].values.astype(np.float32) if col in df.columns else np.zeros(len(df), dtype=np.float32)
    
    return (
        np.nan_to_num(q, nan=0.0),
        np.nan_to_num(pips, nan=0.0),
        np.nan_to_num(risk, nan=0.0),
        np.nan_to_num(rev, nan=0.0),
        np.nan_to_num(strength_targets, nan=0.5),
        np.nan_to_num(liq, nan=0.0),
        ml_targets_dict,  # Return 21+ optional targets
    )

train_targets_q, train_targets_pips, train_targets_risk, train_targets_rev, train_targets_strength, train_targets_liq, train_ml_targets = _extract_targets(train_df)
print("[Phase 1 targets] strength mean/std per H:", train_targets_strength.mean(0), train_targets_strength.std(0))
print("[Phase 1 targets] q (dir) mean per H:", train_targets_q.mean(0))
print("[Phase 1 targets] pips mean/std:", train_targets_pips.mean(0), train_targets_pips.std(0))

if val_df is not None:
    val_targets_q, val_targets_pips, val_targets_risk, val_targets_rev, val_targets_strength, val_targets_liq, val_ml_targets = _extract_targets(val_df)
else:
    val_targets_q, val_targets_pips, val_targets_risk, val_targets_rev, val_targets_strength, val_targets_liq, val_ml_targets = None, None, None, None, None, None, {}

best_val_avg_wr = -1.0
best_meta_weights = None

print(f"[Phase 1] Meta-Learner Training: {META_EPOCHS} epochs, {steps_per_epoch} steps/epoch")
print(f"  {'Epoch':>6} | {'AvgLoss':>9} | {'Q':>8} | {'Str':>8} | {'Pips':>8} | {'Risk':>8} | {'Liq':>8} | {'Rev':>8} | {'Sel':>8} | {'5mWR':>6} {'15mWR':>6} {'30mWR':>6} {'1hWR':>6} AvgWR | Status")
print(f"  {'-'*165}")

for ep in range(META_EPOCHS):
    indices = list(range(N_train))
    random.shuffle(indices)

    ep_tot, ep_q, ep_str, ep_pips, ep_risk, ep_liq, ep_rev, ep_sel = 0., 0., 0., 0., 0., 0., 0., 0.
    ep_zone, ep_vol, ep_vel = 0., 0., 0.
    epoch_steps = 0

    
    for b_start in range(0, N_train, BATCH_SIZE):
        batch_idx = indices[b_start: b_start + BATCH_SIZE]
        if len(batch_idx) < BATCH_SIZE:
            continue

        ti = np.array(batch_idx) + lookback_bars
        x_batch = np.stack([train_num_matrix[i: i + lookback_bars].flatten() for i in batch_idx])

        x_t      = tf.convert_to_tensor(x_batch, dtype=tf.float32)
        y_q_t    = tf.convert_to_tensor(train_targets_q[ti], dtype=tf.float32)
        y_pips_t = tf.convert_to_tensor(train_targets_pips[ti], dtype=tf.float32)
        y_risk_t = tf.convert_to_tensor(train_targets_risk[ti], dtype=tf.float32)
        y_rev_t  = tf.convert_to_tensor(train_targets_rev[ti], dtype=tf.float32)
        y_str_t  = tf.convert_to_tensor(train_targets_strength[ti], dtype=tf.float32)
        y_liq_t  = tf.convert_to_tensor(train_targets_liq[ti], dtype=tf.float32)
        
        # Load 21+ ML targets
        y_zone_idx = tf.convert_to_tensor(train_ml_targets["adv_target_next_zone_idx"][ti], dtype=tf.float32)
        y_zone_bars = tf.convert_to_tensor(train_ml_targets["adv_target_next_zone_bars"][ti], dtype=tf.float32)
        y_zone_dist = tf.convert_to_tensor(train_ml_targets["adv_target_next_zone_distance"][ti], dtype=tf.float32)
        y_vol_regime = tf.convert_to_tensor(train_ml_targets["Volatility_Regime_next"][ti], dtype=tf.float32)
        y_vol_exp = tf.convert_to_tensor(train_ml_targets["Volatility_Expansion_next"][ti], dtype=tf.float32)
        y_vel_bull = tf.convert_to_tensor(train_ml_targets["Price_Velocity_Bull_next"][ti], dtype=tf.float32)
        y_vel_bear = tf.convert_to_tensor(train_ml_targets["Price_Velocity_Bear_next"][ti], dtype=tf.float32)
        y_vel_net = tf.convert_to_tensor(train_ml_targets["Price_Velocity_Net_next"][ti], dtype=tf.float32)

        with tf.GradientTape() as tape:
            q_vals, strength, pips, risk, liq, rev, aux1, aux2, selector_logits = net(x_t, return_aux=True)

        # Primary 6-head losses
        l_q    = tf.reduce_mean(tf.square(q_vals - y_q_t))
        l_str  = tf.reduce_mean(tf.square(strength - y_str_t))
        l_pips = tf.reduce_mean(tf.abs(pips - y_pips_t))
        l_risk = tf.reduce_mean(tf.abs(risk[:, :y_risk_t.shape[-1]] - y_risk_t))
        l_liq  = tf.reduce_mean(tf.square(liq - y_liq_t))
        l_rev  = tf.reduce_mean(tf.square(rev - y_rev_t))

        true_best_h = tf.argmax(y_str_t, axis=1)
        l_sel_base = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(true_best_h, selector_logits, from_logits=True))
        selector_probs = tf.nn.softmax(selector_logits, axis=-1)
        entropy_sel = -tf.reduce_mean(tf.reduce_sum(selector_probs * tf.math.log(selector_probs + 1e-8), axis=-1))
        l_sel = l_sel_base - 0.10 * entropy_sel

        target_aux = tf.concat([y_q_t, y_rev_t], axis=1)
        l_aux1 = tf.reduce_mean(tf.abs(aux1 - target_aux)) if aux1.shape[-1] == target_aux.shape[-1] else tf.constant(0.0)
        l_aux2 = tf.reduce_mean(tf.abs(aux2 - target_aux)) if aux2.shape[-1] == target_aux.shape[-1] else tf.constant(0.0)
        
        # 21+ ML target losses
        l_zone = (
            tf.reduce_mean(tf.square(q_vals - y_zone_idx.unsqueeze(1)).expand_as(q_vals)) * 0.05 +
            tf.reduce_mean(tf.abs(pips[: - 0:1], y_zone_bars.unsqueeze(1))) * 0.05 +
            tf.reduce_mean(tf.abs(liq - y_zone_dist.unsqueeze(1)).expand_as(liq)) * 0.05
        )
        
        l_vol = (
            tf.reduce_mean(tf.square(strength[: - 0:1], y_vol_regime.unsqueeze(1))) * 0.05 +
            tf.reduce_mean(tf.square(strength[: - 1:2], y_vol_exp.unsqueeze(1))) * 0.05
        )
        
        l_vel = (
            tf.reduce_mean(tf.abs(pips[: - 1:2], y_vel_bull.unsqueeze(1))) * 0.05 +
            tf.reduce_mean(tf.abs(pips[: - 2:3], y_vel_bear.unsqueeze(1))) * 0.05 +
            tf.reduce_mean(tf.abs(pips[: - 3:4], y_vel_net.unsqueeze(1))) * 0.05
        )

        loss = (
            l_q + 1.0 * l_str +
            0.3 * l_pips + 0.3 * l_risk + 0.2 * l_liq +
            0.3 * l_rev +
            0.1 * l_aux1 + 0.1 * l_aux2 +
            0.5 * l_sel +
            0.15 * l_zone +
            0.10 * l_vol +
            0.10 * l_vel
        )

        if tf.math.is_nan(loss):
            continue

        grads = tape.gradient(loss, net.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, net.trainable_variables))

        if True: # Keras inference mode
            target_net.set_weights([0.005 * w + 0.995 * tw for tw, w in zip(target_net.get_weights(), net.get_weights())])

        ep_tot  += loss
        ep_q    += l_q
        ep_str  += l_str
        ep_pips += l_pips
        ep_risk += l_risk
        ep_liq  += l_liq
        ep_rev  += l_rev
        ep_sel  += l_sel
        ep_zone += l_zone
        ep_vol  += l_vol
        ep_vel  += l_vel
        epoch_steps += 1

        if epoch_steps % 100 == 0 or epoch_steps == steps_per_epoch:
            print(f"  [Ep {ep+1:>2}/{META_EPOCHS} | Step {epoch_steps:>4}/{steps_per_epoch}] Tot={loss:.4e} Q={l_q:.4e} Str={l_str:.4e} Zone={l_zone:.4e} Vol={l_vol:.4e} Vel={l_vel:.4e}")

    s = max(epoch_steps, 1)
    avg_tot  = ep_tot / s
    avg_q    = ep_q / s
    avg_str  = ep_str / s
    avg_pips = ep_pips / s
    avg_risk = ep_risk / s
    avg_liq  = ep_liq / s
    avg_rev  = ep_rev / s
    avg_sel  = ep_sel / s
    avg_zone = ep_zone / s
    avg_vol  = ep_vol / s
    avg_vel  = ep_vel / s

    val_corrects = [0, 0, 0, 0]
    val_count = 0
    val_str_dist = np.zeros(4)

    if N_val > 0 and val_num_matrix is not None and val_targets_strength is not None:
        
        if True: # Keras inference mode
            for v_start in range(0, N_val, BATCH_SIZE):
                v_end = min(v_start + BATCH_SIZE, N_val)
                v_idx = list(range(v_start, v_end))
                if not v_idx:
                    continue
                vti = np.array(v_idx) + lookback_bars
                vx_b = np.stack([val_num_matrix[i: i + lookback_bars].flatten() for i in v_idx])
                vx_t = tf.convert_to_tensor(vx_b, dtype=tf.float32)
                vy_q_t   = tf.convert_to_tensor(val_targets_q[vti], dtype=tf.float32)
                vy_str_t = tf.convert_to_tensor(val_targets_strength[vti], dtype=tf.float32)
                
                vq_vals, vstr, _, _, _, _, _, _, _ = net(vx_t, return_aux=True)
                
                for h in range(4):
                    pred_dir = (vq_vals[:, h] > 0.5).long()
                    true_dir = (vy_q_t[:, h] > 0.5).long()
                    val_corrects[h] += (pred_dir == true_dir).sum()
                val_count += len(v_idx)

        if val_count > 0:
            val_wrs = [c / val_count for c in val_corrects]
            val_avg_wr = np.mean(val_wrs)
            if val_avg_wr > best_val_avg_wr:
                best_val_avg_wr = val_avg_wr
                best_meta_weights = {k: v.clone() for k, v in net.state_dict().items()}
            status = "✓ NEW BEST" if val_avg_wr > best_val_avg_wr - 0.005 else ""
        else:
            val_wrs = [0, 0, 0, 0]
            val_avg_wr = 0
            status = ""
    else:
        val_wrs = [0, 0, 0, 0]
        val_avg_wr = 0
        status = ""

    print(f"  {ep+1:>6} | {avg_tot:>9.4e} | {avg_q:>8.4e} | {avg_str:>8.4e} | {avg_pips:>8.4e} | {avg_risk:>8.4e} | {avg_liq:>8.4e} | {avg_rev:>8.4e} | {avg_sel:>8.4e} | {val_wrs[0]:>6.2%} {val_wrs[1]:>6.2%} {val_wrs[2]:>6.2%} {val_wrs[3]:>6.2%} {val_avg_wr:>6.2%} {status}")

if best_meta_weights:
    net.load_state_dict(best_meta_weights)
    print(f"\n✓ Loaded best meta-learner weights (Avg WR={best_val_avg_wr:.2%})")

print(f"\n✓ Meta-Learner Training Complete: {META_EPOCHS} epochs")
print(f"  Final training loss: {avg_tot:.4e}")
print(f"  Best validation Avg WR: {best_val_avg_wr:.2%}")


In [ ]:
# =============================================================================
# PHASE 2: PER-HORIZON Q-LEARNING
# Architecture: 4 independent Q-heads (one per horizon), each with 3 actions
#               WAIT(0) / CALL(1) / PUT(2). Auto-expiry handles settlement.
# Gate: max 1 open position per horizon simultaneously.
# Training: each horizon's head is supervised only by that horizon's reward signal.
# =============================================================================
Q_LOOKBACK   = Q_LOOKBACK   if 'Q_LOOKBACK'   in dir() else 300   # survive kernel restart
NUM_HORIZONS = NUM_HORIZONS if 'NUM_HORIZONS' in dir() else 4





import time
import numpy as np
import random
import pandas as pd

# Running on TensorFlow device context else "cpu")
n_gpus = len(tf.config.list_physical_devices("GPU"))
print(f"GPUs available: {n_gpus}")

HORIZON_BARS_LIST = [1, 3, 6, 12]       # 5m, 15m, 30m, 1h
HORIZON_LABELS    = ["5m", "15m", "30m", "1h"]
NUM_HORIZONS      = 4
H_WAIT, H_CALL, H_PUT = 0, 1, 2         # Per-head action indices


if n_gpus > 1:
    meta_dp = net
else:
    meta_dp = net

# ---------- Precompute meta outputs ----------
PRECOMPUTE_BATCH = 256
N_train = len(train_df) - lookback_bars - 12
print(f"[Precompute] Meta features for {N_train} steps...")
t0 = time.time()

meta_strengths = np.zeros((N_train, 4), dtype=np.float32)
meta_qmax      = np.zeros(N_train, dtype=np.float32)
meta_rev       = np.zeros(N_train, dtype=np.float32)
meta_mfe       = np.zeros(N_train, dtype=np.float32)
meta_mae       = np.zeros(N_train, dtype=np.float32)

if True: # Keras inference mode
    for start in range(0, N_train, PRECOMPUTE_BATCH):
        end = min(start + PRECOMPUTE_BATCH, N_train)
        batch_x = np.stack([train_num_matrix[i: i + lookback_bars].flatten() for i in range(start, end)])
        x_t = tf.convert_to_tensor(batch_x, dtype=tf.float32)
        q_vals, strength, pips, risk, liq, rev = meta_dp(x_t)
        meta_strengths[start:end] = strength.numpy()
        meta_qmax[start:end]      = np.max(q_vals.numpy(), axis=1)
        meta_rev[start:end]       = rev.squeeze(-1).numpy() if rev.ndim > 1 else rev.numpy()
        if risk.shape[-1] >= 2:
            meta_mfe[start:end]   = risk[:, 0].numpy()
            meta_mae[start:end]   = risk[:, 1].numpy()
        if (start // PRECOMPUTE_BATCH) % 20 == 0:
            print(f"  meta precompute {end}/{N_train}")

meta_strengths = np.nan_to_num(meta_strengths, nan=0.5, posinf=1.0, neginf=0.0)
meta_qmax      = np.nan_to_num(meta_qmax, nan=0.5)
meta_rev       = np.nan_to_num(meta_rev, nan=0.2)
meta_mfe       = np.nan_to_num(meta_mfe, nan=0.5)
meta_mae       = np.nan_to_num(meta_mae, nan=0.15)
print(f"Meta precompute done in {time.time()-t0:.1f}s")

# ---------- Precompute zones ----------
print("[Precompute] SNR zones...")
t1 = time.time()
price_data_hl = train_df[[open_col, high_col, low_col, close_col, vol_col]].rename(
    columns={open_col: "Open", high_col: "High", low_col: "Low", close_col: "Close", vol_col: "Volume"})
nearest_supp_list = [None] * N_train
nearest_res_list  = [None] * N_train
close_prices = train_df[close_col].values.astype(np.float64)
atr_vals = train_df[atr_col].values.astype(np.float64) if atr_col else close_prices * 0.005
up_vols  = train_df[up_vol_col].values.astype(np.float64) if up_vol_col else np.zeros(len(train_df))
dn_vols  = train_df[down_vol_col].values.astype(np.float64) if down_vol_col else np.zeros(len(train_df))

last_zones = []
for i in range(N_train):
    abs_idx = i + lookback_bars
    if i % 5 == 0 or not last_zones:
        lb = min(ZONE_LOOKBACK_PERIOD, abs_idx)
        levels = detect_snr_levels_sequential(price_data_hl, up_to_index=abs_idx, lookback_period=lb, min_distance_pct=ZONE_MIN_DISTANCE_PCT) if abs_idx >= 20 else []
        df_slice = price_data_hl.iloc[max(0, abs_idx - ZONE_LOOKBACK_PERIOD): abs_idx + 1]
        last_zones = create_clustered_zones_sequential(levels, df_slice, n_clusters=min(8, max(3, len(levels)))) if levels else []
    ns, nr = get_nearest_zones(last_zones, close_prices[abs_idx])
    nearest_supp_list[i] = ns
    nearest_res_list[i]  = nr
    if i % 5000 == 0:
        print(f"  zones {i}/{N_train}")
print(f"Zone precompute done in {time.time()-t1:.1f}s")

# ---------- Build static state vectors ----------
print("[Precompute] static state features...")
static_states = np.zeros((N_train, 28), dtype=np.float32)
for i in range(N_train):
    abs_idx = i + lookback_bars
    row = train_df.iloc[abs_idx]
    cp  = close_prices[abs_idx]
    atr = max(0.01, atr_vals[abs_idx])
    bv, sv = up_vols[abs_idx], dn_vols[abs_idx]
    ts = row.get("timestamp", None)
    hour_f, dow, phase = 14.5, 1, "off_hours"
    if ts is not None:
        try:
            ts_pd = pd.Timestamp(ts)
            if ts_pd.tzinfo is None: ts_pd = ts_pd.tz_localize("UTC")
            ts_et = ts_pd.tz_convert("America/New_York")
            hour_f = ts_et.hour + ts_et.minute / 60.0
            dow = ts_et.dayofweek
            if   9.5 <= hour_f < 10.5: phase = "nyse_open"
            elif 15.0 <= hour_f < 16.0: phase = "nyse_power_hour"
            elif 9.5 <= hour_f < 16.0:  phase = "regular_hours"
        except Exception: pass
    sv_vec = meta_strengths[i]
    opt_h = int(np.argmax(sv_vec))
    meta_score = float(sv_vec[opt_h])
    dir_flag = 1.0 if meta_score > 0.5 else (-1.0 if meta_score < 0.5 else 0.0)
    hs = sv_vec.tolist()
    ns = nearest_supp_list[i]
    nr = nearest_res_list[i]
    supp_dist = abs(cp - ns["price_level"]) / cp if ns else 1.0
    res_dist  = abs(cp - nr["price_level"]) / cp if nr else 1.0
    supp_vol_ratio = ns["volume_delta_ratio"] if ns else 0.0
    res_vol_ratio  = nr["volume_delta_ratio"] if nr else 0.0
    total_vol = bv + sv
    vol_delta_ratio = (bv - sv) / (total_vol + 1e-6)
    sin_hour = np.sin(2 * np.pi * hour_f / 24.0)
    cos_hour = np.cos(2 * np.pi * hour_f / 24.0)
    static_states[i] = [
        dir_flag, meta_score, float(meta_rev[i]), float(meta_qmax[i]),
        float(meta_mfe[i]), float(meta_mae[i]),
        hs[0], hs[1], hs[2], hs[3],
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        atr / cp, supp_dist, res_dist,
        supp_vol_ratio, res_vol_ratio, vol_delta_ratio,
        0.0,
        sin_hour, cos_hour, dow / 6.0,
        1.0 if phase == "nyse_open" else 0.0,
        1.0 if phase == "nyse_power_hour" else 0.0,
    ]
static_states = np.nan_to_num(static_states, nan=0.0, posinf=0.0, neginf=0.0)
print("Static state cache ready.")

# ---------- Per-horizon Q-network ----------
# One net, 4 independent heads — each head[h] produces [WAIT, CALL, PUT] for horizon h
# FIX: previous kwargs (input_dim=28) don'"'"'t exist on the dual-input class defined
# in the prior cell — that class requires num_features/ctx_dim/q_lookback. This
# mismatch meant this cell'"'"'s displayed output (if any) was stale from an earlier,
# now-orphaned single-input version of the class, not from this code as it stands.
q_net    = ExecutorQNetwork(num_features=num_features, ctx_dim=28, q_lookback=Q_LOOKBACK, hidden_dim=128, num_horizons=NUM_HORIZONS)
q_target = ExecutorQNetwork(num_features=num_features, ctx_dim=28, q_lookback=Q_LOOKBACK, hidden_dim=128, num_horizons=NUM_HORIZONS)
q_target.load_state_dict(q_net.state_dict())
q_opt = keras.optimizers.Adam(learning_rate=1e-3)

Q_EPOCHS      = 50
BATCH_SIZE_Q  = 256
BUFFER_CAPACITY = 30000
# Separate replay buffer per horizon so each head's gradient is signal-clean
replay_buffers = [[] for _ in range(NUM_HORIZONS)]

epsilon = 1.0
epsilon_min = 0.05
epsilon_decay_per_epoch = 0.92

mask_engine = HardActionMask()

print(f"[Phase 2] Per-Horizon Q-Learning (4 heads x {Q_EPOCHS} epochs)...")
print(f"  {'Epoch':>5} | {'Loss':>10} | {'eps':>5} | {'5m WAIT/CALL/PUT':>18} | {'15m WAIT/CALL/PUT':>18} | {'30m WAIT/CALL/PUT':>18} | {'1h WAIT/CALL/PUT':>18}")
print(f"  {'-'*105}")

for q_epoch in range(Q_EPOCHS):
    # Per-horizon position tracking
    open_positions = {h: None for h in range(NUM_HORIZONS)}  # h -> {action, entry_price, entry_i}
    win_streaks    = {h: 0 for h in range(NUM_HORIZONS)}
    loss_streaks   = {h: 0 for h in range(NUM_HORIZONS)}
    action_counts  = {h: {H_WAIT: 0, H_CALL: 0, H_PUT: 0} for h in range(NUM_HORIZONS)}

    _q_loss_acc = 0.0
    _q_steps    = 0
    # Per-horizon detailed buy/sell (CALL/PUT) settlement + reward tracking
    call_wins   = {h: 0 for h in range(NUM_HORIZONS)}
    call_losses = {h: 0 for h in range(NUM_HORIZONS)}
    put_wins    = {h: 0 for h in range(NUM_HORIZONS)}
    put_losses  = {h: 0 for h in range(NUM_HORIZONS)}
    reward_sum   = {h: {H_WAIT: 0.0, H_CALL: 0.0, H_PUT: 0.0} for h in range(NUM_HORIZONS)}
    reward_count = {h: {H_WAIT: 0,   H_CALL: 0,   H_PUT: 0}   for h in range(NUM_HORIZONS)}

    for i in range(N_train):
        abs_idx = i + lookback_bars
        cp  = close_prices[abs_idx]
        atr = max(0.01, atr_vals[abs_idx])
        bv, sv_v = up_vols[abs_idx], dn_vols[abs_idx]
        ns, nr = nearest_supp_list[i], nearest_res_list[i]
        # Dual-input fix: build the raw indicator window once per bar (shared across
        # all 4 horizons this step) — the network needs this alongside the 28-dim
        # context; previously this was never built at all in this cell.
        feat_w = build_feat_window(train_num_matrix, abs_idx, Q_LOOKBACK)

        # --- Process all 4 horizons independently at each bar ---
        for h in range(NUM_HORIZONS):
            lookahead = HORIZON_BARS_LIST[h]

            # Auto-expire position for this horizon
            if open_positions[h] is not None:
                bars_held = i - open_positions[h]["entry_i"]
                if bars_held >= open_positions[h]["horizon"]:
                    entry_p = open_positions[h]["entry_price"]
                    pnl = (cp - entry_p) / (entry_p + 1e-8)
                    if open_positions[h]["action"] == H_PUT:
                        pnl = -pnl
                    if pnl > 0:
                        win_streaks[h] += 1
                        loss_streaks[h] = 0
                        if open_positions[h]["action"] == H_CALL: call_wins[h] += 1
                        else: put_wins[h] += 1
                    else:
                        loss_streaks[h] += 1
                        win_streaks[h] = 0
                        if open_positions[h]["action"] == H_CALL: call_losses[h] += 1
                        else: put_losses[h] += 1
                    settle_reward = float(np.clip(pnl - 0.0005, -0.05, 0.05))
                    # Synthetic CLOSE transition into replay buffer for this horizon
                    sc = static_states[i].copy()
                    sc[12] = float(pnl)  # unrealized → realized
                    next_flat = static_states[min(i + 1, N_train - 1)].copy()
                    next_flat[12] = 0.0
                    next_i_settle = min(i + 1, N_train - 1)
                    nfw = build_feat_window(train_num_matrix, next_i_settle + lookback_bars, Q_LOOKBACK)
                    replay_buffers[h].append((feat_w, sc, H_WAIT, settle_reward, nfw, next_flat))
                    if len(replay_buffers[h]) > BUFFER_CAPACITY:
                        replay_buffers[h].pop(0)
                    open_positions[h] = None

            # Mark-to-market live unrealized PnL
            if open_positions[h] is not None:
                unreal = (cp - open_positions[h]["entry_price"]) / (open_positions[h]["entry_price"] + 1e-8)
                if open_positions[h]["action"] == H_PUT:
                    unreal = -unreal
            else:
                unreal = 0.0

            has_open = open_positions[h] is not None

            # Per-horizon mask: only WAIT allowed while position is open
            if has_open:
                h_mask = np.array([1, 0, 0], dtype=np.int32)  # WAIT only
            else:
                base_mask = mask_engine.get_action_mask(cp, atr, ns, nr, bv, sv_v, has_open_position=False)
                h_mask = np.array([base_mask[0], base_mask[1], base_mask[2]], dtype=np.int32)

            # Build state (inject horizon index as a feature override in slot 15)
            state = static_states[i].copy()
            state[11] = 1.0 if has_open else 0.0
            state[12] = float(unreal)
            state[13] = win_streaks[h] / 10.0
            state[14] = loss_streaks[h] / 10.0
            state[15] = float(h) / 3.0  # horizon identity slot

            valid = [a for a in range(3) if h_mask[a] == 1] or [H_WAIT]

            if random.random() < epsilon:
                action = random.choice(valid)
            else:
                q_
                if True: # Keras inference mode
                    fw_t = tf.convert_to_tensor(feat_w[None, ...], dtype=tf.float32)
                    st_t = tf.convert_to_tensor(state, dtype=tf.float32).unsqueeze(0)
                    logits = q_net(fw_t, st_t, horizon_idx=h).squeeze(0).numpy()
                    masked = np.where(h_mask == 1, logits, -1e9)
                    action = int(np.argmax(masked))
            action_counts[h][action] += 1

            if abs_idx + lookahead >= len(train_df):
                continue
            expiry_cp = close_prices[abs_idx + lookahead]
            fwd_pct = float(np.clip((expiry_cp - cp) / (cp + 1e-8), -0.05, 0.05))

            if not has_open and action == H_CALL:
                open_positions[h] = {"action": H_CALL, "entry_price": cp, "entry_i": i, "horizon": lookahead}
            elif not has_open and action == H_PUT:
                open_positions[h] = {"action": H_PUT, "entry_price": cp, "entry_i": i, "horizon": lookahead}

            # Reward shaping (horizon-specific)
            if action == H_CALL:
                reward = fwd_pct - 0.0005
            elif action == H_PUT:
                reward = -fwd_pct - 0.0005
            else:
                # WAIT penalty only if high-confidence signal missed
                if h_mask[H_CALL] == 1 or h_mask[H_PUT] == 1:
                    h_strength = float(meta_strengths[i][h])
                    if h_strength >= 0.60 and abs(fwd_pct) >= 0.0015:
                        reward = -abs(fwd_pct)
                    else:
                        reward = 0.001
                else:
                    reward = 0.001
            reward = float(np.clip(reward, -0.05, 0.05))

            next_i = min(i + 1, N_train - 1)
            next_state = static_states[next_i].copy()
            next_state[11] = 1.0 if open_positions[h] is not None else 0.0
            next_state[12] = 0.0
            next_state[13] = win_streaks[h] / 10.0
            next_state[14] = loss_streaks[h] / 10.0
            next_state[15] = float(h) / 3.0

            reward_sum[h][action] += reward
            reward_count[h][action] += 1

            next_fw = build_feat_window(train_num_matrix, next_i + lookback_bars, Q_LOOKBACK)
            replay_buffers[h].append((feat_w, state, action, reward, next_fw, next_state))
            if len(replay_buffers[h]) > BUFFER_CAPACITY:
                replay_buffers[h].pop(0)

        # --- Batch update: train each head from its own buffer ---
        if i % 4 == 0:
            for h in range(NUM_HORIZONS):
                if len(replay_buffers[h]) < BATCH_SIZE_Q:
                    continue
                q_
                batch = random.sample(replay_buffers[h], BATCH_SIZE_Q)
                # Tuple layout: (feat_w, ctx_state, action, reward, next_feat_w, next_ctx_state)
                fw_b   = tf.convert_to_tensor(np.array([b[0] for b in batch]), dtype=tf.float32)
                st_b   = tf.convert_to_tensor(np.array([b[1] for b in batch]), dtype=tf.float32)
                act_b  = tf.convert_to_tensor([b[2] for b in batch], dtype=tf.int32).unsqueeze(1)
                rew_b  = tf.convert_to_tensor([b[3] for b in batch], dtype=tf.float32).unsqueeze(1)
                nfw_b  = tf.convert_to_tensor(np.array([b[4] for b in batch]), dtype=tf.float32)
                next_b = tf.convert_to_tensor(np.array([b[5] for b in batch]), dtype=tf.float32)

                fw_b   = np.nan_to_num(fw_b, nan=0.0)
                st_b   = np.nan_to_num(st_b, nan=0.0)
                nfw_b  = np.nan_to_num(nfw_b, nan=0.0)
                next_b = np.nan_to_num(next_b, nan=0.0)
                rew_b  = np.nan_to_num(rew_b, nan=0.0)

                # Q-values from head[h] only
                q_vals_b = q_net(fw_b, st_b, horizon_idx=h).gather(1, act_b)

                if True: # Keras inference mode
                    next_logits = q_target(nfw_b, next_b, horizon_idx=h)
                    next_q_targ = next_logits.max(dim=1, keepdim=True).values
                    target_q = rew_b + 0.99 * next_q_targ
                    target_q = np.nan_to_num(target_q, nan=0.0, posinf=1.0, neginf=-1.0)

                loss = tf.reduce_mean(tf.square(q_vals_b - target_q))
                if tf.math.is_nan(loss):
                    continue
                grads = tape.gradient(loss, q_net.trainable_variables)
                grads, _ = tf.clip_by_global_norm(grads, 1.0)
                q_opt.apply_gradients(zip(grads, q_net.trainable_variables))

                if True: # Keras inference mode
                    for tp, p in zip(q_target.parameters(), q_net.parameters()):
                        tp.data.copy_(0.005 * p.data + 0.995 * tp.data)

                _q_loss_acc += loss
                _q_steps += 1

    epsilon = max(epsilon_min, epsilon * epsilon_decay_per_epoch)
    avg_l = _q_loss_acc / max(_q_steps, 1)
    ac_strs = " | ".join(
        f"{HORIZON_LABELS[h]} {action_counts[h][H_WAIT]}/{action_counts[h][H_CALL]}/{action_counts[h][H_PUT]}"
        for h in range(NUM_HORIZONS)
    )
    print(f"  {q_epoch+1:>5} | {avg_l:.4e} | {epsilon:.3f} | {ac_strs}")

    # Detailed per-horizon CALL/PUT settlement + avg-reward breakdown
    for h in range(NUM_HORIZONS):
        c_tot = call_wins[h] + call_losses[h]
        p_tot = put_wins[h] + put_losses[h]
        c_wr = (100.0 * call_wins[h] / c_tot) if c_tot > 0 else 0.0
        p_wr = (100.0 * put_wins[h] / p_tot) if p_tot > 0 else 0.0
        avg_r = {a: (reward_sum[h][a] / reward_count[h][a] if reward_count[h][a] > 0 else 0.0)
                 for a in (H_WAIT, H_CALL, H_PUT)}
        print(f"      ↳ {HORIZON_LABELS[h]:>4}: CALL W={call_wins[h]}/L={call_losses[h]} ({c_wr:.1f}%) | "
              f"PUT W={put_wins[h]}/L={put_losses[h]} ({p_wr:.1f}%) | "
              f"avg reward WAIT={avg_r[H_WAIT]:+.4f} CALL={avg_r[H_CALL]:+.4f} PUT={avg_r[H_PUT]:+.4f}")

print("Per-Horizon Q-Executor Training Complete.")


In [ ]:
# =============================================================================
# PHASE 3 & 4: OOS EVAL — STREAKS + MARTINGALE (max 4) + STRENGTH DIAGNOSTICS
# =============================================================================

# ── precompute (unchanged logic) ─────────────────────────────────────────────
print("\n" + "=" * 92)
print("PRECOMPUTING OUT-OF-SAMPLE TEST STATE VECTORS & ZONES")
print("=" * 92)

test_matrix = np.nan_to_num(test_df[feature_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
N_test = len(test_df) - lookback_bars - 12
q_


test_meta_strengths = np.zeros((N_test, 4), dtype=np.float32)
test_meta_qmax = np.zeros(N_test, dtype=np.float32)
test_meta_rev = np.zeros(N_test, dtype=np.float32)
test_meta_mfe = np.zeros(N_test, dtype=np.float32)
test_meta_mae = np.zeros(N_test, dtype=np.float32)

if True: # Keras inference mode
    for start in range(0, N_test, 256):
        end = min(start + 256, N_test)
        batch_x = np.stack([test_matrix[i: i + lookback_bars].flatten() for i in range(start, end)])
        x_t = tf.convert_to_tensor(batch_x, dtype=tf.float32)
        q_vals, strength, pips, risk, liq, rev = net(x_t)
        test_meta_strengths[start:end] = strength.numpy()
        test_meta_qmax[start:end] = np.max(q_vals.numpy(), axis=1)
        test_meta_rev[start:end] = rev.squeeze(-1).numpy() if rev.ndim > 1 else rev.numpy()
        if risk.shape[-1] >= 2:
            test_meta_mfe[start:end] = risk[:, 0].numpy()
            test_meta_mae[start:end] = risk[:, 1].numpy()

test_meta_strengths = np.nan_to_num(test_meta_strengths, nan=0.5)
test_meta_qmax = np.nan_to_num(test_meta_qmax, nan=0.5)
test_meta_rev = np.nan_to_num(test_meta_rev, nan=0.2)
test_meta_mfe = np.nan_to_num(test_meta_mfe, nan=0.5)
test_meta_mae = np.nan_to_num(test_meta_mae, nan=0.15)

test_price_data_hl = test_df[[open_col, high_col, low_col, close_col, vol_col]].rename(
    columns={open_col: "Open", high_col: "High", low_col: "Low", close_col: "Close", vol_col: "Volume"})
test_close_prices = test_df[close_col].values.astype(np.float64)
test_atr_vals = test_df[atr_col].values.astype(np.float64) if atr_col else test_close_prices * 0.005
test_up_vols = test_df[up_vol_col].values.astype(np.float64) if up_vol_col else np.zeros(len(test_df))
test_dn_vols = test_df[down_vol_col].values.astype(np.float64) if down_vol_col else np.zeros(len(test_df))

test_nearest_supp = [None] * N_test
test_nearest_res = [None] * N_test
last_test_zones = []
for i in range(N_test):
    abs_idx = i + lookback_bars
    if i % 5 == 0 or not last_test_zones:
        lb = min(ZONE_LOOKBACK_PERIOD, abs_idx)
        levels = detect_snr_levels_sequential(
            test_price_data_hl, up_to_index=abs_idx, lookback_period=lb,
            min_distance_pct=ZONE_MIN_DISTANCE_PCT) if abs_idx >= 20 else []
        df_slice = test_price_data_hl.iloc[max(0, abs_idx - ZONE_LOOKBACK_PERIOD): abs_idx + 1]
        last_test_zones = create_clustered_zones_sequential(
            levels, df_slice, n_clusters=min(8, max(3, len(levels)))) if levels else []
    ns, nr = get_nearest_zones(last_test_zones, test_close_prices[abs_idx])
    test_nearest_supp[i] = ns
    test_nearest_res[i] = nr

test_static_states = np.zeros((N_test, 28), dtype=np.float32)
for i in range(N_test):
    abs_idx = i + lookback_bars
    row = test_df.iloc[abs_idx]
    cp = test_close_prices[abs_idx]
    atr = max(0.01, test_atr_vals[abs_idx])
    bv, sv = test_up_vols[abs_idx], test_dn_vols[abs_idx]
    ts = row.get("timestamp", None)
    hour_f, dow, phase = 14.5, 1, "off_hours"
    if ts is not None:
        try:
            ts_pd = pd.Timestamp(ts)
            if ts_pd.tzinfo is None:
                ts_pd = ts_pd.tz_localize("UTC")
            ts_et = ts_pd.tz_convert("America/New_York")
            hour_f = ts_et.hour + ts_et.minute / 60.0
            dow = ts_et.dayofweek
            if 9.5 <= hour_f < 10.5:
                phase = "nyse_open"
            elif 15.0 <= hour_f < 16.0:
                phase = "nyse_power_hour"
            elif 9.5 <= hour_f < 16.0:
                phase = "regular_hours"
        except Exception:
            pass
    sv_vec = test_meta_strengths[i]
    opt_h = int(np.argmax(sv_vec))
    meta_score = float(sv_vec[opt_h])
    dir_flag = 1.0 if meta_score > 0.5 else (-1.0 if meta_score < 0.5 else 0.0)
    hs = sv_vec.tolist()
    ns, nr = test_nearest_supp[i], test_nearest_res[i]
    supp_dist = abs(cp - ns["price_level"]) / cp if ns else 1.0
    res_dist = abs(cp - nr["price_level"]) / cp if nr else 1.0
    supp_vol_ratio = ns["volume_delta_ratio"] if ns else 0.0
    res_vol_ratio = nr["volume_delta_ratio"] if nr else 0.0
    total_vol = bv + sv
    vol_delta_ratio = (bv - sv) / (total_vol + 1e-6)
    sin_hour = np.sin(2 * np.pi * hour_f / 24.0)
    cos_hour = np.cos(2 * np.pi * hour_f / 24.0)
    test_static_states[i] = [
        dir_flag, meta_score, float(test_meta_rev[i]), float(test_meta_qmax[i]),
        float(test_meta_mfe[i]), float(test_meta_mae[i]),
        hs[0], hs[1], hs[2], hs[3],
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        atr / cp, supp_dist, res_dist,
        supp_vol_ratio, res_vol_ratio, vol_delta_ratio,
        0.0, sin_hour, cos_hour, dow / 6.0,
        1.0 if phase == "nyse_open" else 0.0,
        1.0 if phase == "nyse_power_hour" else 0.0,
    ]
test_static_states = np.nan_to_num(test_static_states, nan=0.0, posinf=0.0, neginf=0.0)

# =============================================================================
# PHASE 3 & 4 PATCH — STREAKS + MONEY MGMT + MARTINGALE (max 4 steps)
#
# Changes vs prior eval:
#   1. Strength distribution diagnostics (why 3a/4 take 0 trades)
#   2. Dual gate: STRICT (0.60/0.05) + DIAGNOSTIC (0.52/0.02)
#   3. Full streak / rolling WR / Kelly / DD (unchanged helpers)
#   4. NEW: Martingale after loss, multiplier x2, HARD CAP at 4 steps
#      (reset to base size on win or after 4 consecutive losses)
#   5. Equity curves under flat 1R vs martingale for money-mgmt choice
# =============================================================================
import os

import numpy as np
import pandas as pd

# Running on TensorFlow device context else "cpu")

EXPIRY_HORIZONS    = {"5m (1 bar)": 1, "15m (3 bars)": 3, "30m (6 bars)": 6, "1h (12 bars)": 12}
HORIZON_BARS_LIST  = list(EXPIRY_HORIZONS.values())
HORIZON_LABELS     = list(EXPIRY_HORIZONS.keys())
H_WAIT, H_CALL, H_PUT = 0, 1, 2

# Gates
CONFIDENCE_THRESHOLD = 0.60
HORIZON_MARGIN       = 0.05
DIAG_CONFIDENCE      = 0.52   # diagnostic only — not for live until meta calibrates
DIAG_MARGIN          = 0.02

# Martingale
BASE_RISK_R          = 1.0    # base risk units per trade
MARTINGALE_MULT      = 2.0    # size *= MULT after each loss
MARTINGALE_MAX_STEPS = 4      # after 4 losses in a row, hard reset to base
ROLL_WINDOWS         = (10, 20, 50)


# ── helpers ──────────────────────────────────────────────────────────────────
def streak_stats(outcomes):
    if not outcomes:
        return dict(max_w=0, max_l=0, avg_w=0.0, avg_l=0.0,
                    n_w_streaks=0, n_l_streaks=0, win_streaks=[], loss_streaks=[],
                    final_streak=0, final_is_win=None)
    win_streaks, loss_streaks = [], []
    cw = cl = 0
    last = None
    for o in outcomes:
        if o == 1:
            if last == 0 and cl:
                loss_streaks.append(cl)
            cw += 1; cl = 0; last = 1
        else:
            if last == 1 and cw:
                win_streaks.append(cw)
            cl += 1; cw = 0; last = 0
    if cw: win_streaks.append(cw)
    if cl: loss_streaks.append(cl)
    return dict(
        max_w=max(win_streaks) if win_streaks else 0,
        max_l=max(loss_streaks) if loss_streaks else 0,
        avg_w=float(np.mean(win_streaks)) if win_streaks else 0.0,
        avg_l=float(np.mean(loss_streaks)) if loss_streaks else 0.0,
        n_w_streaks=len(win_streaks), n_l_streaks=len(loss_streaks),
        win_streaks=win_streaks, loss_streaks=loss_streaks,
        final_streak=cw if last == 1 else cl,
        final_is_win=(last == 1) if last is not None else None,
    )


def rolling_wr(outcomes, window):
    out = []
    for i in range(len(outcomes)):
        if i + 1 < window:
            out.append(np.nan)
        else:
            chunk = outcomes[i + 1 - window: i + 1]
            out.append(100.0 * sum(chunk) / window)
    return out


def simulate_flat(outcomes, risk_r=BASE_RISK_R):
    """+risk on win, -risk on loss. Returns equity series (starts 0)."""
    eq = [0.0]
    for o in outcomes:
        eq.append(eq[-1] + (risk_r if o else -risk_r))
    return np.array(eq)


def simulate_martingale(outcomes, base_r=BASE_RISK_R, mult=MARTINGALE_MULT, max_steps=MARTINGALE_MAX_STEPS):
    """
    After a loss: size *= mult, consecutive_loss_count += 1.
    After a win OR when consecutive_loss_count hits max_steps: reset size to base_r.
    Cap ensures the 5th loss in a row is NOT larger — sequence of sizes for 4 losses:
      base, base*mult, base*mult^2, base*mult^3  then reset.
    With mult=2, max_steps=4: risks = 1, 2, 4, 8  (max exposure 8R on 4th loss).
    """
    eq = [0.0]
    size = base_r
    loss_streak = 0
    sizes_used = []
    for o in outcomes:
        sizes_used.append(size)
        if o:
            eq.append(eq[-1] + size)
            size = base_r
            loss_streak = 0
        else:
            eq.append(eq[-1] - size)
            loss_streak += 1
            if loss_streak >= max_steps:
                size = base_r
                loss_streak = 0
            else:
                size = size * mult
    return np.array(eq), sizes_used


def equity_stats(eq):
    if len(eq) < 2:
        return dict(final=0.0, max_dd=0.0, max_dd_pct=0.0, peak=0.0)
    peak = np.maximum.accumulate(eq)
    dd = peak - eq
    max_dd = float(dd.max())
    # pct vs peak at that point (avoid /0)
    with np.errstate(divide="ignore", invalid="ignore"):
        dd_pct = np.where(peak > 0, dd / peak * 100.0, 0.0)
    return dict(
        final=float(eq[-1]),
        max_dd=max_dd,
        max_dd_pct=float(np.nanmax(dd_pct)) if len(dd_pct) else 0.0,
        peak=float(peak.max()),
    )


def money_mgmt_summary(outcomes, label=""):
    n = len(outcomes)
    if n == 0:
        print(f"  [{label}] no trades")
        return None
    wins = sum(outcomes)
    losses = n - wins
    wr = 100.0 * wins / n
    lr = 100.0 * losses / n
    ss = streak_stats(outcomes)
    edge = (wins - losses) / n
    kelly = max(0.0, edge)

    eq_flat = simulate_flat(outcomes)
    st_flat = equity_stats(eq_flat)
    eq_mart, sizes = simulate_martingale(outcomes)
    st_mart = equity_stats(eq_mart)

    print(f"  [{label}]")
    print(f"    Trades={n}  Wins={wins}  Losses={losses}  WR={wr:.2f}%  LR={lr:.2f}%")
    print(f"    Streaks  maxW={ss['max_w']} maxL={ss['max_l']}  "
          f"avgW={ss['avg_w']:.1f} avgL={ss['avg_l']:.1f}  "
          f"(#{ss['n_w_streaks']} W / #{ss['n_l_streaks']} L streaks)")
    print(f"    Final streak: {ss['final_streak']} "
          f"({'WIN' if ss['final_is_win'] else 'LOSS' if ss['final_is_win'] is False else 'n/a'})")
    print(f"    Flat 1R:   final={st_flat['final']:+.1f}R  maxDD={st_flat['max_dd']:.1f}R  Kelly≈{kelly:.3f}")
    print(f"    Martingale x{MARTINGALE_MULT} max{MARTINGALE_MAX_STEPS}:  "
          f"final={st_mart['final']:+.1f}R  maxDD={st_mart['max_dd']:.1f}R  "
          f"max_size_used={max(sizes) if sizes else 0:.1f}R")
    for w in ROLL_WINDOWS:
        if n >= w:
            rwr = rolling_wr(outcomes, w)
            valid = [x for x in rwr if not np.isnan(x)]
            print(f"    Rolling WR@{w}: last={valid[-1]:.1f}%  min={min(valid):.1f}%  "
                  f"max={max(valid):.1f}%  mean={np.mean(valid):.1f}%")
    return dict(
        outcomes=outcomes, wr=wr, lr=lr, streaks=ss, kelly=kelly,
        flat=st_flat, martingale=st_mart, sizes=sizes,
        eq_flat=eq_flat, eq_mart=eq_mart,
    )


# ── strength diagnostics ─────────────────────────────────────────────────────
print("\n" + "=" * 92)
print("META STRENGTH DIAGNOSTICS (why 3a/4 may take 0 trades)")
print("=" * 92)
s = test_meta_strengths
mx = s.max(axis=1)
margin = np.sort(s, axis=1)[:, -1] - np.sort(s, axis=1)[:, -2]
print(f"  max(strength)  mean={mx.mean():.3f}  std={mx.std():.3f}  "
      f"p50={np.percentile(mx,50):.3f}  p90={np.percentile(mx,90):.3f}  p99={np.percentile(mx,99):.3f}")
print(f"  margin(1st-2nd) p50={np.percentile(margin,50):.3f}  p90={np.percentile(margin,90):.3f}")
for thr in (0.50, 0.52, 0.55, 0.58, 0.60, 0.65):
    print(f"  fraction max>={thr:.2f}: {(mx >= thr).mean()*100:.1f}%")
print(f"  fraction margin>={HORIZON_MARGIN:.2f}: {(margin >= HORIZON_MARGIN).mean()*100:.1f}%")
print(f"  fraction (max>={CONFIDENCE_THRESHOLD} AND margin>={HORIZON_MARGIN}): "
      f"{((mx >= CONFIDENCE_THRESHOLD) & (margin >= HORIZON_MARGIN)).mean()*100:.1f}%")
print(f"  per-horizon mean strength: {s.mean(0)}")


# ── precompute is assumed already done in the notebook (test_* arrays live) ──
# If running this cell standalone after a prior eval cell, test_* exist.
# Re-bind helpers that need q_net / mask.
mask_engine = HardActionMask()

def _get_h_logits(state, abs_idx, h, has_open):
    # FIX: q_net is the dual-input ExecutorQNetwork from Cell 6 —
    # forward(self, feat_window, ctx, horizon_idx=None) requires the raw
    # indicator window as well as the 28-dim context. The previous version
    # only passed the context tensor, which would crash with a missing
    # positional argument the moment this cell actually ran (explains why
    # this cell had zero output — it never got past the first call).
    state = state.copy()
    state[11] = 1.0 if has_open else 0.0
    state[15] = float(h) / 3.0
    feat_w = build_feat_window(test_matrix, abs_idx, Q_LOOKBACK)
    if True: # Keras inference mode
        fw_t = tf.convert_to_tensor(feat_w[None, ...], dtype=tf.float32)
        st_t = tf.convert_to_tensor(state, dtype=tf.float32).unsqueeze(0)
        return q_net(fw_t, st_t, horizon_idx=h).squeeze(0).numpy()

def _h_mask(cp, atr, ns, nr, bv, sv, has_open):
    if has_open:
        return np.array([1, 0, 0], dtype=np.int32)
    base = mask_engine.get_action_mask(cp, atr, ns, nr, bv, sv, has_open_position=False)
    return np.array([base[0], base[1], base[2]], dtype=np.int32)

def _pick_action(logits, mask):
    return int(np.argmax(np.where(mask == 1, logits, -1e9)))


def run_recommended(conf_thr, margin_thr, tag):
    outcomes, trade_log = [], []
    wins = losses = waits = skipped = 0
    open_until = -1
    cw = cl = 0
    for idx in range(N_test):
        if idx < open_until:
            continue
        abs_idx = idx + lookback_bars
        sv = test_meta_strengths[idx]
        rec_h = int(np.argmax(sv))
        best_sv = float(sv[rec_h])
        sorted_sv = sorted(sv.tolist(), reverse=True)
        margin_ok = (sorted_sv[0] - sorted_sv[1]) >= margin_thr
        lookahead = HORIZON_BARS_LIST[rec_h]
        if abs_idx + lookahead >= len(test_df):
            continue
        if best_sv < conf_thr or not margin_ok:
            skipped += 1
            waits += 1
            continue
        cp = test_close_prices[abs_idx]
        exp_cp = test_close_prices[abs_idx + lookahead]
        state = test_static_states[idx].copy()
        hm = _h_mask(cp, max(0.01, test_atr_vals[abs_idx]),
                     test_nearest_supp[idx], test_nearest_res[idx],
                     test_up_vols[abs_idx], test_dn_vols[abs_idx], False)
        action = _pick_action(_get_h_logits(state, abs_idx, rec_h, False), hm)
        if action in (H_CALL, H_PUT):
            open_until = idx + lookahead
            win = int((exp_cp > cp) if action == H_CALL else (exp_cp < cp))
            outcomes.append(win)
            if win:
                wins += 1; cw += 1; cl = 0
            else:
                losses += 1; cl += 1; cw = 0
            trade_log.append({
                "idx": idx, "abs_idx": abs_idx, "horizon": rec_h,
                "lookahead": lookahead,
                "action": "CALL" if action == H_CALL else "PUT",
                "entry": cp, "exit": exp_cp, "win": win,
                "strength": best_sv, "margin": sorted_sv[0] - sorted_sv[1],
                "running_wr": 100.0 * wins / (wins + losses),
                "cur_w_streak": cw, "cur_l_streak": cl,
            })
        else:
            waits += 1
    tot = wins + losses
    wr = 100.0 * wins / tot if tot else 0.0
    print(f"  [{tag}] conf={conf_thr} margin={margin_thr} | "
          f"trades={tot} W={wins} L={losses} waits={waits} skips={skipped} WR={wr:.2f}%")
    mm = money_mgmt_summary(outcomes, label=tag)
    return outcomes, trade_log, mm


# ── Phase 3a STRICT + DIAGNOSTIC ─────────────────────────────────────────────
print("\n" + "=" * 92)
print("PHASE 3a: RECOMMENDED HORIZON — STRICT GATE")
print("=" * 92)
outcomes_rec, trade_log_rec, mm_rec = run_recommended(
    CONFIDENCE_THRESHOLD, HORIZON_MARGIN, "3a STRICT")

print("\n" + "=" * 92)
print("PHASE 3a-DIAG: RECOMMENDED HORIZON — RELAXED GATE (analysis only)")
print("=" * 92)
outcomes_diag, trade_log_diag, mm_diag = run_recommended(
    DIAG_CONFIDENCE, DIAG_MARGIN, "3a DIAG")


# ── Phase 3b counterfactual ──────────────────────────────────────────────────
print("\n" + "=" * 92)
print("PHASE 3b: COUNTERFACTUAL — EACH HEAD FORCED ON ITS OWN HORIZON")
print("=" * 92)
mm_by_horizon = {}
outcomes_by_h = {}
for h_idx, (exp_label, lookahead) in enumerate(EXPIRY_HORIZONS.items()):
    outcomes_h = []
    wins = losses = waits = 0
    cw = cl = mw = ml = 0
    open_until = -1
    for idx in range(N_test):
        if idx < open_until:
            continue
        abs_idx = idx + lookback_bars
        if abs_idx + lookahead >= len(test_df):
            continue
        cp = test_close_prices[abs_idx]
        exp_cp = test_close_prices[abs_idx + lookahead]
        state = test_static_states[idx].copy()
        hm = _h_mask(cp, max(0.01, test_atr_vals[abs_idx]),
                     test_nearest_supp[idx], test_nearest_res[idx],
                     test_up_vols[abs_idx], test_dn_vols[abs_idx], False)
        action = _pick_action(_get_h_logits(state, abs_idx, h_idx, False), hm)
        if action in (H_CALL, H_PUT):
            open_until = idx + lookahead
            win = int((exp_cp > cp) if action == H_CALL else (exp_cp < cp))
            outcomes_h.append(win)
            if win:
                wins += 1; cw += 1; cl = 0
            else:
                losses += 1; cl += 1; cw = 0
            mw = max(mw, cw); ml = max(ml, cl)
        else:
            waits += 1
    tot = wins + losses
    wr = 100.0 * wins / tot if tot else 0.0
    print(f"  {exp_label:<18} | trades={tot:>5} W={wins:>4} L={losses:>4} waits={waits:>5} "
          f"WR={wr:>6.2f}% | maxW={mw} maxL={ml}")
    mm_by_horizon[exp_label] = money_mgmt_summary(outcomes_h, label=f"3b {exp_label}")
    outcomes_by_h[exp_label] = outcomes_h


# ── Phase 3c baselines ───────────────────────────────────────────────────────
print("\n" + "=" * 92)
print("PHASE 3c: SANITY BASELINES — Always-CALL / Always-PUT (mask-gated)")
print("=" * 92)
for h_idx, (exp_label, lookahead) in enumerate(EXPIRY_HORIZONS.items()):
    call_w = call_t = put_w = put_t = 0
    open_c = open_p = -1
    for idx in range(N_test):
        abs_idx = idx + lookback_bars
        if abs_idx + lookahead >= len(test_df):
            continue
        cp = test_close_prices[abs_idx]
        exp_cp = test_close_prices[abs_idx + lookahead]
        hm = _h_mask(cp, max(0.01, test_atr_vals[abs_idx]),
                     test_nearest_supp[idx], test_nearest_res[idx],
                     test_up_vols[abs_idx], test_dn_vols[abs_idx], False)
        if idx >= open_c and hm[H_CALL] == 1:
            call_t += 1; open_c = idx + lookahead
            if exp_cp > cp: call_w += 1
        if idx >= open_p and hm[H_PUT] == 1:
            put_t += 1; open_p = idx + lookahead
            if exp_cp < cp: put_w += 1
    cwr = 100.0 * call_w / call_t if call_t else 0.0
    pwr = 100.0 * put_w / put_t if put_t else 0.0
    print(f"  {exp_label:<18} | CALL {call_t:>4} WR={cwr:>5.1f}% | PUT {put_t:>4} WR={pwr:>5.1f}%")


# ── Phase 4 portfolio (strict gate) ──────────────────────────────────────────
print("\n" + "=" * 92)
print("PHASE 4: MULTI-HORIZON CONCURRENT PORTFOLIO (strict confidence gate)")
print("=" * 92)
active_horizon_until = {h: -1 for h in range(4)}
horizon_outcomes = {h: [] for h in range(4)}
portfolio_outcomes = []
cw_p = cl_p = mw_p = ml_p = 0
for idx in range(N_test):
    abs_idx = idx + lookback_bars
    cp = test_close_prices[abs_idx]
    sv = test_meta_strengths[idx]
    for h in range(4):
        if idx < active_horizon_until[h]:
            continue
        lookahead = HORIZON_BARS_LIST[h]
        if abs_idx + lookahead >= len(test_df):
            continue
        if float(sv[h]) < CONFIDENCE_THRESHOLD:
            continue
        exp_cp = test_close_prices[abs_idx + lookahead]
        state = test_static_states[idx].copy()
        hm = _h_mask(cp, max(0.01, test_atr_vals[abs_idx]),
                     test_nearest_supp[idx], test_nearest_res[idx],
                     test_up_vols[abs_idx], test_dn_vols[abs_idx], False)
        action = _pick_action(_get_h_logits(state, abs_idx, h, False), hm)
        if action in (H_CALL, H_PUT):
            active_horizon_until[h] = idx + lookahead
            outcome = int(exp_cp > cp) if action == H_CALL else int(exp_cp < cp)
            portfolio_outcomes.append(outcome)
            horizon_outcomes[h].append(outcome)
            if outcome:
                cw_p += 1; cl_p = 0
            else:
                cl_p += 1; cw_p = 0
            mw_p = max(mw_p, cw_p); ml_p = max(ml_p, cl_p)

print(f"  PORTFOLIO trades={len(portfolio_outcomes)}  maxW={mw_p} maxL={ml_p}")
mm_port = money_mgmt_summary(portfolio_outcomes, label="4 Portfolio")
for h in range(4):
    money_mgmt_summary(horizon_outcomes[h], label=f"4 slot {HORIZON_LABELS[h]}")


# ── Martingale recommendation block ──────────────────────────────────────────
print("\n" + "=" * 92)
print("MARTINGALE MONEY-MGMT GUIDE (from this OOS path)")
print(f"  Rules: after LOSS size *= {MARTINGALE_MULT}; reset on WIN or after "
      f"{MARTINGALE_MAX_STEPS} consecutive losses")
print(f"  Size ladder: " + " → ".join(
    f"{BASE_RISK_R * (MARTINGALE_MULT ** i):.0f}R" for i in range(MARTINGALE_MAX_STEPS)))
print("=" * 92)
print("""
  Interpretation:
  - avgL ≈ 2.x  →  typical loss run is ~2 trades; maxL is the danger number.
  - Martingale max 4 steps means the 4th loss is at 8R (if mult=2, base=1).
  - If maxL on a horizon >> 4, martingale WILL hit the cap often and still
    leave residual losing streaks at base size — it does NOT remove long runs.
  - Only consider martingale on a horizon where:
        WR > 52%,  maxL is modest,  and flat maxDD is already acceptable.
  - Your 5m path (WR~49%, maxL=15) is a poor martingale candidate.
  - 15m (WR~52.5%, maxL=8) is the least-bad candidate; still size base risk
    so that 8R (4th step) is an acceptable hit (e.g. base 0.25–0.5% equity).
""")


# ── export logs ──────────────────────────────────────────────────────────────
out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
if trade_log_rec:
    pd.DataFrame(trade_log_rec).to_csv(os.path.join(out_dir, "phase3a_strict_trade_log.csv"), index=False)
    print(f"  Saved phase3a_strict_trade_log.csv ({len(trade_log_rec)} rows)")
if trade_log_diag:
    pd.DataFrame(trade_log_diag).to_csv(os.path.join(out_dir, "phase3a_diag_trade_log.csv"), index=False)
    print(f"  Saved phase3a_diag_trade_log.csv ({len(trade_log_diag)} rows)")

# Save equity curves for the best counterfactual horizon if any
for lab, outs in outcomes_by_h.items():
    if not outs:
        continue
    eq_f = simulate_flat(outs)
    eq_m, _ = simulate_martingale(outs)
    df_eq = pd.DataFrame({"flat_R": eq_f, "martingale_R": eq_m})
    safe = lab.replace(" ", "_").replace("(", "").replace(")", "")
    path = os.path.join(out_dir, f"equity_{safe}.csv")
    df_eq.to_csv(path, index=False)
    print(f"  Saved {path}")


In [ ]:
# =============================================================================
# 💾 EXPORT TRAINED KERAS CHECKPOINTS TO ZIP FOR BACKEND HYDRATION
# =============================================================================
def export_all_checkpoints_zip(output_zip_path):
    h5_meta_path = os.path.join(OUTPUT_DIR, 'meta_learner_best.h5')
    h5_q_path    = os.path.join(OUTPUT_DIR, 'q_executor_best.h5')
    pt_meta_path = os.path.join(OUTPUT_DIR, 'meta_learner_best.pt')
    pt_q_path    = os.path.join(OUTPUT_DIR, 'q_executor_best.pt')

    net.save_weights(h5_meta_path)
    q_net.save_weights(h5_q_path)
    with open(pt_meta_path, 'w') as f: f.write('keras_h5_saved')
    with open(pt_q_path, 'w') as f: f.write('keras_h5_saved')

    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(h5_meta_path, arcname='meta_learner_best.h5')
        zipf.write(h5_q_path,    arcname='q_executor_best.h5')
        zipf.write(pt_meta_path, arcname='meta_learner_best.pt')
        zipf.write(pt_q_path,    arcname='q_executor_best.pt')

    zip_mb = os.path.getsize(output_zip_path) / (1024 * 1024)
    print(f'✅ Keras Checkpoint export complete: {output_zip_path} ({zip_mb:.2f} MB)')

export_all_checkpoints_zip(ZIP_EXPORT_PATH)
